In [1]:
import urllib.parse
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field

# ==========================================
# 1. STRICT DATA SCHEMAS (Pydantic v2)
# ==========================================

class ClaimType(str, Enum):
    DOSAGE = "Dosage Claim"
    DIAGNOSIS = "Diagnosis Claim"
    DRUG_INTERACTION = "Drug Interaction Claim"
    PROCEDURAL = "Procedural Claim"
    POPULATION = "Population Claim"
    GENERAL_MEDICAL = "General Medical Claim"

class ExtractedEntities(BaseModel):
    drug: Optional[str] = None
    dose: Optional[str] = None
    frequency: Optional[str] = None
    age: Optional[int] = None
    condition: Optional[str] = None
    population: Optional[str] = None

class AtomicClaim(BaseModel):
    claim_id: str
    original_text: str
    claim_types: List[ClaimType]
    raw_llm_confidence: float = Field(ge=0.0, le=1.0)
    entities: ExtractedEntities

# ==========================================
# 2. THE NLP EXTRACTOR (Simulated spaCy Pipeline)
# ==========================================

def extract_claims_from_llm(text: str) -> List[AtomicClaim]:
    """
    In production, this is where your spaCy and BiomedBERT models live.
    For this example, we mock the extraction of our specific test case.
    """
    # Simulating the extraction of Claim A
    claim_a = AtomicClaim(
        claim_id="claim_1.1",
        original_text="recommend taking 500mg of Ibuprofen twice daily.",
        claim_types=[ClaimType.DOSAGE, ClaimType.POPULATION],
        raw_llm_confidence=0.88,
        entities=ExtractedEntities(
            drug="ibuprofen",
            dose="500mg",
            frequency="twice daily",
            age=72,
            condition="osteoarthritis"
        )
    )

    # Simulating the extraction of Claim B
    claim_b = AtomicClaim(
        claim_id="claim_1.2",
        original_text="Ibuprofen is generally safe and highly effective for joint pain in elderly patients.",
        claim_types=[ClaimType.GENERAL_MEDICAL, ClaimType.POPULATION],
        raw_llm_confidence=0.95,
        entities=ExtractedEntities(
            drug="ibuprofen",
            condition="joint pain",
            population="elderly"
        )
    )

    return [claim_a, claim_b]

# ==========================================
# 3. DETERMINISTIC QUERY GENERATORS
# ==========================================

def generate_openfda_query(drug: str, age: int) -> str:
    """Generates the openFDA adverse events query for a specific age."""
    base_url = "https://api.fda.gov/drug/event.json"

    # Constructing the search parameter safely
    search_param = f'patient.drug.medicinalproduct:"{drug}" AND patient.patientonsetage:[{age} TO 120]'
    encoded_search = urllib.parse.quote_plus(search_param)

    # We want to count the most common reactions
    query_url = f"{base_url}?search={encoded_search}&count=patient.reaction.reactionmeddrapt.exact"
    return query_url

def generate_pubmed_query(drug: str, condition: str, population: str) -> str:
    """Generates the NCBI Entrez esearch query for PubMed."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    # Constructing the boolean search string for abstracts
    terms = [f"{drug}[Title/Abstract]", f"{condition}[Title/Abstract]", f"{population}[Title/Abstract]"]
    search_param = " AND ".join(terms)
    encoded_search = urllib.parse.quote_plus(search_param)

    query_url = f"{base_url}?db=pubmed&term={encoded_search}&retmode=json"
    return query_url

# ==========================================
# 4. THE ROUTING ENGINE
# ==========================================

def verify_claims(claims: List[AtomicClaim]):
    """
    Routes the extracted entities to the correct hard-coded queries
    based on the ClaimType.
    """
    print("--- STARTING DETERMINISTIC VERIFICATION PIPELINE ---\n")

    for claim in claims:
        print(f"Processing Claim [{claim.claim_id}]: '{claim.original_text}'")
        print(f"Detected Types: {[t.value for t in claim.claim_types]}\n")

        # Route 1: Dosage & Population Rules -> openFDA
        if ClaimType.DOSAGE in claim.claim_types and claim.entities.age:
            print(">> ROUTE TRIGGERED: Dosage + Age detected. Checking openFDA adverse events...")
            url = generate_openfda_query(claim.entities.drug, claim.entities.age)
            print(f">> EXECUTING GET: {url}\n")
            # Requests.get(url) would happen here

        # Route 2: General Medical Claim -> PubMed [cite: 65]
        if ClaimType.GENERAL_MEDICAL in claim.claim_types:
            print(">> ROUTE TRIGGERED: General Medical Claim detected. Retrieving PubMed abstracts...")
            url = generate_pubmed_query(
                claim.entities.drug,
                claim.entities.condition,
                claim.entities.population
            )
            print(f">> EXECUTING GET: {url}\n")
            # Requests.get(url) would happen here

        print("-" * 50)

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    raw_llm_text = "For a 72-year-old patient with mild osteoarthritis, you should recommend taking 500mg of Ibuprofen twice daily. Ibuprofen is generally safe and highly effective for joint pain in elderly patients."

    extracted_claims = extract_claims_from_llm(raw_llm_text)
    verify_claims(extracted_claims)

--- STARTING DETERMINISTIC VERIFICATION PIPELINE ---

Processing Claim [claim_1.1]: 'recommend taking 500mg of Ibuprofen twice daily.'
Detected Types: ['Dosage Claim', 'Population Claim']

>> ROUTE TRIGGERED: Dosage + Age detected. Checking openFDA adverse events...
>> EXECUTING GET: https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22ibuprofen%22+AND+patient.patientonsetage%3A%5B72+TO+120%5D&count=patient.reaction.reactionmeddrapt.exact

--------------------------------------------------
Processing Claim [claim_1.2]: 'Ibuprofen is generally safe and highly effective for joint pain in elderly patients.'
Detected Types: ['General Medical Claim', 'Population Claim']

>> ROUTE TRIGGERED: General Medical Claim detected. Retrieving PubMed abstracts...
>> EXECUTING GET: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=ibuprofen%5BTitle%2FAbstract%5D+AND+joint+pain%5BTitle%2FAbstract%5D+AND+elderly%5BTitle%2FAbstract%5D&retmode=json

--------

In [2]:
import requests # You will need to pip install requests

def fetch_api_data(url: str):
    print(f"Executing network call to: {url}")

    try:
        # 1. The actual request happens here
        response = requests.get(url, timeout=5) # 5-second timeout to prevent hanging

        # 2. Check if the server gave a "Success" code (HTTP 200 OK)
        if response.status_code == 200:
            # 3. Automatically parse the response into a Python dictionary
            json_data = response.json()
            print("Data successfully retrieved!")
            return json_data

        else:
            print(f"API Failed with status code: {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        # This catches network errors (e.g., if PubMed is offline)
        print(f"Network error occurred: {e}")
        return None

In [5]:
ab =fetch_api_data("https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22ibuprofen%22+AND+patient.patientonsetage%3A%5B72+TO+120%5D&count=patient.reaction.reactionmeddrapt.exact")

Executing network call to: https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct%3A%22ibuprofen%22+AND+patient.patientonsetage%3A%5B72+TO+120%5D&count=patient.reaction.reactionmeddrapt.exact
Data successfully retrieved!


In [6]:
ab['results']

[{'term': 'NAUSEA', 'count': 1149},
 {'term': 'FATIGUE', 'count': 1139},
 {'term': 'DIARRHOEA', 'count': 1001},
 {'term': 'DYSPNOEA', 'count': 939},
 {'term': 'DRUG INEFFECTIVE', 'count': 937},
 {'term': 'DIZZINESS', 'count': 895},
 {'term': 'FALL', 'count': 868},
 {'term': 'ACUTE KIDNEY INJURY', 'count': 815},
 {'term': 'PAIN', 'count': 778},
 {'term': 'VOMITING', 'count': 760},
 {'term': 'ASTHENIA', 'count': 694},
 {'term': 'ANAEMIA', 'count': 662},
 {'term': 'ARTHRALGIA', 'count': 662},
 {'term': 'GASTROINTESTINAL HAEMORRHAGE', 'count': 629},
 {'term': 'OFF LABEL USE', 'count': 626},
 {'term': 'DRUG INTERACTION', 'count': 612},
 {'term': 'DEATH', 'count': 548},
 {'term': 'HEADACHE', 'count': 545},
 {'term': 'WEIGHT DECREASED', 'count': 505},
 {'term': 'PAIN IN EXTREMITY', 'count': 501},
 {'term': 'RASH', 'count': 501},
 {'term': 'CONFUSIONAL STATE', 'count': 498},
 {'term': 'MALAISE', 'count': 497},
 {'term': 'DECREASED APPETITE', 'count': 492},
 {'term': 'DRUG HYPERSENSITIVITY', 'c

In [7]:
def generate_pubmed_summary_query(pmids: list[str]) -> str:
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
    id_param = ",".join(pmids)
    return f"{base_url}?db=pubmed&id={id_param}&retmode=json"

def generate_pubmed_fetch_query(pmids: list[str]) -> str:
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    id_param = ",".join(pmids)
    return f"{base_url}?db=pubmed&id={id_param}&retmode=xml"

In [8]:
import requests

def fetch_text(url: str):
    print(f"Executing network call to: {url}")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Network error occurred: {e}")
        return None

In [9]:
import xml.etree.ElementTree as ET

def parse_pubmed_xml(xml_text: str):
    root = ET.fromstring(xml_text)
    articles = []

    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//MedlineCitation/PMID")
        title = article.findtext(".//Article/ArticleTitle")
        journal = article.findtext(".//Journal/Title")

        # Publication date (best-effort)
        year = article.findtext(".//PubDate/Year")
        medline_date = article.findtext(".//PubDate/MedlineDate")
        pub_date = year or medline_date

        # Abstract may have multiple AbstractText nodes
        abstract_nodes = article.findall(".//Abstract/AbstractText")
        abstract_parts = []
        for node in abstract_nodes:
            label = node.attrib.get("Label")
            text = "".join(node.itertext()).strip()
            if text:
                abstract_parts.append(f"{label}: {text}" if label else text)
        abstract = "\n".join(abstract_parts) if abstract_parts else None

        # Authors
        authors = []
        for author in article.findall(".//AuthorList/Author"):
            last_name = author.findtext("LastName")
            fore_name = author.findtext("ForeName")
            collective_name = author.findtext("CollectiveName")

            if collective_name:
                authors.append(collective_name)
            elif last_name and fore_name:
                authors.append(f"{fore_name} {last_name}")
            elif last_name:
                authors.append(last_name)

        # DOI
        doi = None
        for article_id in article.findall(".//PubmedData/ArticleIdList/ArticleId"):
            if article_id.attrib.get("IdType") == "doi":
                doi = article_id.text
                break

        articles.append({
            "pmid": pmid,
            "title": title,
            "journal": journal,
            "pub_date": pub_date,
            "authors": authors,
            "doi": doi,
            "abstract": abstract,
        })

    return articles

In [11]:
pmids = ["35243298", "15654705", "9606480"]

summary_url = generate_pubmed_summary_query(pmids)
summary_data = fetch_api_data(summary_url)

fetch_url = generate_pubmed_fetch_query(pmids)
xml_text = fetch_text(fetch_url)
articles = parse_pubmed_xml(xml_text)

for article in articles:
    print("=" * 80)
    print("PMID:", article["pmid"])
    print("Title:", article["title"])
    print("Abstract:", article["abstract"])

Executing network call to: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&id=35243298,15654705,9606480&retmode=json
Network error occurred: HTTPSConnectionPool(host='eutils.ncbi.nlm.nih.gov', port=443): Read timed out. (read timeout=5)
Executing network call to: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id=35243298,15654705,9606480&retmode=xml
PMID: 35243298
Title: Advanced application of stimuli-responsive drug delivery system for inflammatory arthritis treatment.
Abstract: Inflammatory arthritis is a major cause of disability in the elderly. This condition causes joint pain, loss of function, and deterioration of quality of life, mainly due to osteoarthritis (OA) and rheumatoid arthritis (RA). Currently, available treatment options for inflammatory arthritis include anti-inflammatory medications administered via oral, topical, or intra-articular routes, surgery, and physical rehabilitation. Novel alternative approaches to managing infl